# 고속도로 휴게소 충전소 — 실시간 혼잡도 데이터 준비

`streamlit_app.py`의 "근처 충전소 추천"이 지금까지는 카카오 로컬 검색("전기차 충전소" 키워드)으로
후보를 찾고, 이름에 "휴게소"가 들어있는지로 고속도로 접근성을 판별했다. 두 가지 문제가 있었다:

1. **이름 매칭은 부정확하다.** 고속도로 위에서 검색해도 나들목 밖 읍·면사무소 충전소가 섞여 나왔다
   (실제로 추풍령 부근에서 "추풍령면사무소 주차장"이 고속도로 휴게소보다 먼저 추천된 적이 있다).
2. **실시간 혼잡도가 없다.** 카카오 검색 결과는 공공데이터포털 충전소 ID와 연결할 방법이 없어서
   `data.go.kr`(환경부 `B552584/EvCharger`)의 실시간 상태(`stat`)를 붙일 수 없었다.

## 해법

`getChargerInfo`는 위치 반경 검색은 지원하지 않지만(직접 확인함 — `lat`/`lng`/`radius` 파라미터를
줘도 무시되고 전국 결과가 그대로 나온다), **`kind`/`kindDetail` 필터는 실제로 동작한다.**
표본을 뽑아 확인해보니 `kind=C0`(`kindDetail` C001·C002)가 고속도로 휴게소·영업소 충전기와
정확히 일치했다 — 전국 2,486건으로, zcode(시도)별로 수만~수천 건씩 나오던 전체 데이터에 비해
훨씬 다루기 쉬운 크기다.

이 노트북은 `kind=C0` 데이터를 한 번 통째로 받아 위경도 인덱스로 저장한다. 그러면 웹앱은:
- **후보 검색**: 이 로컬 인덱스에서 하버사인 최근접 검색 (API 호출 없음, 빠름)
- **실제 도로거리**: 후보 몇 개에 대해서만 카카오 길찾기 호출
- **실시간 혼잡도**: 후보 몇 개에 대해서만 `statId`로 `getChargerInfo` 재호출 (정확한 매칭, 매번 최신)

이름 매칭도, 대규모 전국 스캔도 필요 없어진다.

## 0. 설정

In [1]:
import os
import time
import urllib.parse

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
DATA_GO_KR_KEY = urllib.parse.unquote(os.getenv("DATA_GO_KR_KEY", ""))
assert DATA_GO_KR_KEY, "DATA_GO_KR_KEY 가 .env 에 없습니다"

EVCHARGER_URL = "https://apis.data.go.kr/B552584/EvCharger/getChargerInfo"
OUT_CSV = "data/highway_rest_area_chargers.csv"


def call(params, retries=3):
    p = {"serviceKey": DATA_GO_KR_KEY, "dataType": "JSON", **params}
    for i in range(retries):
        r = requests.get(EVCHARGER_URL, params=p, timeout=15)
        if r.status_code == 200:
            return r.json()
        time.sleep(1.5 * (i + 1))
    r.raise_for_status()

## 1. `kind=C0` 필터 확인

파라미터로 실제 필터링이 되는지, 전국 규모가 얼마나 되는지 먼저 확인한다.

In [2]:
probe = call({"pageNo": 1, "numOfRows": 1, "kind": "C0"})
total = probe["totalCount"]
print(f"전국 kind=C0(고속도로/자동차전용도로 관련 시설) 총 {total:,}건")
assert total < 20_000, "예상보다 훨씬 큽니다 — kind 코드가 바뀌었을 수 있으니 다시 확인하세요."

전국 kind=C0(고속도로/자동차전용도로 관련 시설) 총 2,486건


## 2. 전국 데이터 수집 (페이지네이션)

In [3]:
PAGE_SIZE = 1000
rows = []
page = 1
while True:
    j = call({"pageNo": page, "numOfRows": PAGE_SIZE, "kind": "C0"})
    items = j.get("items", {}).get("item", [])
    if not items:
        break
    rows.extend(items)
    print(f"  page {page}: {len(items)}건 (누적 {len(rows)})")
    if len(items) < PAGE_SIZE:
        break
    page += 1
    time.sleep(0.2)

raw = pd.DataFrame(rows)
print(f"\n원본 {len(raw)}행 (충전기 커넥터 단위, 충전소당 여러 행)")

  page 1: 1000건 (누적 1000)


  page 2: 1000건 (누적 2000)


  page 3: 486건 (누적 2486)

원본 2486행 (충전기 커넥터 단위, 충전소당 여러 행)


## 3. 충전소 단위로 정리 · 필터링 · 저장

같은 충전소(`statId`)에 커넥터가 여러 개면 여러 행으로 온다. 위경도 인덱스는 충전소 단위면 되니
`statId` 기준으로 대표 1행만 남긴다 (실시간 상태는 앱에서 매번 `statId`로 다시 조회해서
그때 커넥터별 현황을 전부 본다 — 여기 저장하는 상태값은 스냅샷일 뿐 실시간이 아니다).

**`kindDetail`로 한 번 더 거른다.** `kind=C0` 안에는 세 하위코드가 섞여 있다:
- `C001` 휴게소, `C002` 영업소/TG 주차장 — 표본을 찍어보니 전부 실제 고속도로 본선 시설이었다.
- `C003` "내트럭하우스" 계열 — 화물차 전용 시설이라 취지에는 맞지만, 표본 주소(부산 남구 신선로,
  김천시 시청로 등)를 보면 항만·시가지 인근으로 **고속도로 나들목 밖**에 있다. 그대로 두면 이 앱이
  고치려던 문제("고속도로 위인데 여기를 어떻게 가나")가 그대로 재현된다. 그래서 뺀다.

In [4]:
raw["lat"] = raw["lat"].astype(float)
raw["lng"] = raw["lng"].astype(float)

before = raw["statId"].nunique()
raw = raw[raw["kindDetail"].isin(["C001", "C002"])]
after = raw["statId"].nunique()
print(f"kindDetail 필터: 충전소 {before}개 -> {after}개 (C003 내트럭하우스 {before - after}개 제외)")

stations = (
    raw.sort_values("chgerId")
    .drop_duplicates(subset="statId", keep="first")
    [["statId", "statNm", "addr", "lat", "lng", "kindDetail", "busiNm"]]
    .reset_index(drop=True)
)

os.makedirs("data", exist_ok=True)
stations.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"충전소 {len(stations)}개 (커넥터 {len(raw)}개) -> {OUT_CSV} 저장 완료")
stations.sample(5, random_state=42)

kindDetail 필터: 충전소 789개 -> 670개 (C003 내트럭하우스 119개 제외)
충전소 670개 (커넥터 2237개) -> data/highway_rest_area_chargers.csv 저장 완료


,statId,statNm,addr,lat,lng,kindDetail,busiNm
361,EC470009,남성주참외휴게소 (창원방향) (급),경상북도 성주군 선남면 중부내륙고속도로 77,35.869590,128.316700,C001,이지차저
158,ME21D011,장안(울산) 휴게소,부산광역시 기장군 장안읍 동해고속도로 26,35.381878,129.248577,C001,기후에너지환경부
480,SGMAC003,이인휴게소(천안방향),충청남도 공주시 이인면 논산천안고속도로 32,37.266160,127.404788,C001,SK시그넷
640,ME22A354,옥산(부산)휴게소,충청북도 청주시 흥덕구 옥산면 오산리 590-15,36.657039,127.370340,C001,기후에너지환경부
275,CV000927,생곡휴게소,충청북도 괴산군 칠성면 연풍로 114,36.776971,127.891891,C002,채비


## 4. 검증 — 알던 지점과 대조

In [5]:
def haversine_km(lng1, lat1, lng2, lat2):
    import numpy as np
    lng1, lat1, lng2, lat2 = (np.radians(v) for v in (lng1, lat1, lng2, lat2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2)
    return 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


# 추풍령(경부고속도로, 충북/경북 경계) 근처에서 최근접 5곳 — 앞서 실제로 문제가 됐던 지점
probe_lat, probe_lng = 36.207, 127.900
d = stations.copy()
d["dist_km"] = haversine_km(probe_lng, probe_lat, d["lng"], d["lat"])
print(d.sort_values("dist_km")[["statNm", "addr", "dist_km"]].head(5).to_string(index=False))

    statNm                       addr  dist_km
황간(서울) 휴게소 충청북도 영동군 황간면 회포1길 65 (회포리) 6.150630
 황간(서울)휴게소           충청북도 영동군 회포1길 65 6.150630
 황간(서울)휴게소           충청북도 영동군 회포1길 65 6.150630
황간(서울) 휴게소       충청북도 영동군 황간면 회포리 117 6.167457
황간휴게소(부산)        충청북도 영동군 황간면 회포길 102 6.352211
